In [ ]:
from transformers import pipeline

In [ ]:
image_to_text = pipeline("image-to-text", model="nlpconnect/vit-gpt2-image-captioning")

Could not find image processor class in the image processor config or the model config. Loading based on pattern matching with the model's feature extractor configuration.
/usr/local/lib/python3.10/dist-packages/transformers/models/vit/feature_extraction_vit.py:28: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


In [ ]:
caption = image_to_text("/content/sample_image.jpg")[0]['generated_text']

In [ ]:
print(caption)

a woman is sitting at a table with two men 


# Emoji

In [ ]:
!unzip /content/drive/MyDrive/Project/emoji/emojify.zip

Archive:  /content/drive/MyDrive/Project/emoji/emojify.zip
replace emojify_data.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: ㅛ
error:  invalid response [ㅛ]
replace emojify_data.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: ㅛ
error:  invalid response [ㅛ]
replace emojify_data.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: emojify_data.csv        
replace test_emoji.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: test_emoji.csv          
replace train_emoji.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: train_emoji.csv         


In [ ]:
!unzip /content/drive/MyDrive/Project/emoji/glove.zip

Archive:  /content/drive/MyDrive/Project/emoji/glove.zip
replace glove.6B.50d.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: glove.6B.50d.txt        


In [ ]:
!pip install emoji

In [ ]:
import emoji
import numpy as np
import pandas as pd

## Text Preprocessing using Embeddings

In [ ]:
glove_file = open("/content/glove.6B.50d.txt", encoding = 'utf8')

In [ ]:
def intialize_emb_matrix(file):
    embedding_matrix = {}
    for line in file:
        values = line.split()
        word = values[0]
        embedding = np.array(values[1:], dtype='float64')
        embedding_matrix[word] = embedding

    return embedding_matrix


In [ ]:
embedding_matrix = intialize_emb_matrix(glove_file)

In [ ]:
embedding_matrix['dance']

array([-9.9716e-01,  6.1348e-01, -1.7686e+00,  8.7052e-02, -2.4644e-01,
        1.9984e-01, -2.5892e-01, -3.1309e-02, -4.3117e-01,  1.0844e+00,
        4.7032e-01, -3.6731e-01,  1.0047e-01,  1.2459e+00,  2.5728e-03,
       -3.8805e-01,  4.8489e-01,  3.8052e-01, -5.6983e-01, -8.4332e-01,
        8.8607e-01,  9.7396e-01,  9.0600e-03,  9.5187e-01,  3.3867e-01,
       -2.5218e-01, -1.5633e+00, -4.8921e-01, -3.4712e-01, -1.2629e+00,
        2.8455e+00,  2.8887e-01, -1.9158e-01,  6.5195e-02, -1.9799e-01,
        2.2815e-01,  4.4613e-01, -8.2659e-02, -3.3068e-01,  4.8144e-02,
       -2.7224e-01, -2.1765e-01, -5.5931e-01, -4.0149e-01,  5.6395e-01,
       -1.9915e-01,  7.2645e-01, -1.5184e+00, -8.7863e-01,  2.1860e-01])

In [ ]:
embedding_matrix['dance'].shape

(50,)

In [ ]:
def get_emb_data(data, max_len):
#     max_len = 168
    embedding_data = np.zeros((len(data), max_len, 50))  # from glove6B50d

    for idx in range(data.shape[0]):
        words_in_sentence = data[idx].split()

        for i in range(len(words_in_sentence)):
            if embedding_matrix.get(words_in_sentence[i].lower()) is not None:
                embedding_data[idx][i] = embedding_matrix[words_in_sentence[i].lower()]

    return embedding_data

## emoji dataset

In [ ]:
train_data = pd.read_csv("/content/train_emoji.csv", header=None)
test_data = pd.read_csv("/content/test_emoji.csv", header=None)

In [ ]:
train_data.head()

,0,1,2,3
0,never talk to me again,3,NaN,NaN
1,I am proud of your achievements,2,NaN,NaN
2,It is the worst day in my life,3,NaN,NaN
3,Miss you so much,0,NaN,[0]
4,food is life,4,NaN,NaN


In [ ]:
test_data.head()

,0,1
0,I want to eat\t,4
1,he did not answer\t,3
2,he got a very nice raise\t,2
3,she got me a nice present\t,2
4,ha ha ha it was so funny\t,2


In [ ]:
emoji.emojize(':angry_face:')

'😠'

In [ ]:
emoji_mapping = {
    '0': ':beating_heart:',
    '1': ':baseball:',
    '2': ':beaming_face_with_smiling_eyes:',
    '3': ':angry_face:',
    '4': ':face_savoring_food:'
}

In [ ]:
for key, value in emoji_mapping.items():
    emoji_mapping[key] = emoji.emojize(value)

print(emoji_mapping)


{'0': '💓', '1': '⚾', '2': '😁', '3': '😠', '4': '😋'}


In [ ]:
X_train = train_data[0].values
y_train = train_data[1].values

In [ ]:
X_train = get_emb_data(X_train, 10)

In [ ]:
X_train

array([[[ 9.5387e-02, -1.6865e-01, -1.1514e-01, ..., -4.2103e-01,
         -5.3817e-01,  1.3738e-01],
        [-3.4604e-02,  4.1087e-01, -9.7368e-02, ...,  5.6204e-01,
         -5.9770e-01,  9.5068e-01],
        [ 6.8047e-01, -3.9263e-02,  3.0186e-01, ..., -7.3297e-02,
         -6.4699e-02, -2.6044e-01],
        ...,
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00, ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00, ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00, ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00]],

       [[ 1.1891e-01,  1.5255e-01, -8.2073e-02, ..., -5.7512e-01,
         -2.6671e-01,  9.2121e-01],
        [ 3.4664e-01,  3.9805e-01,  4.8970e-01, ...,  8.6576e-02,
          3.4037e-01,  1.3588e+00],
        [-5.9180e-01,  2.7671e-01, -4.6971e-01, ..., -2.0055e-01,
         -2.6487e-01,  5.7981e-01],
        ...,
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00, ...,  

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import LSTM, SimpleRNN, Dense, Dropout

In [ ]:
from keras.utils import to_categorical

In [ ]:
y_train = to_categorical(y_train)
y_train

array([[0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.],
       [1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [1., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 1., 0., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 1., 0.],
       [0., 1., 0., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 1., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.],
       [1., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 1.],
       [1., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 1., 0., 0.],
       [1., 0.

In [ ]:
model_tte = Sequential()
model_tte.add(LSTM(units=64, input_shape=(10, 50), return_sequences=True))
model_tte.add(Dropout(0.3))
model_tte.add(LSTM(units=32))
model_tte.add(Dropout(0.2))
model_tte.add(Dense(units=10, activation='relu'))
model_tte.add(Dense(units=5, activation='softmax'))

In [ ]:
model_tte.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_6 (LSTM)               (None, 10, 64)            29440     
                                                                 
 dropout_6 (Dropout)         (None, 10, 64)            0         
                                                                 
 lstm_7 (LSTM)               (None, 32)                12416     
                                                                 
 dropout_7 (Dropout)         (None, 32)                0         
                                                                 
 dense_8 (Dense)             (None, 10)                330       
                                                                 
 dense_9 (Dense)             (None, 5)                 55        
                                                                 
Total params: 42241 (165.00 KB)
Trainable params: 4224

In [ ]:
model_tte.compile(optimizer='adam', loss=keras.losses.categorical_crossentropy, metrics=['acc'])

In [ ]:
history = model_tte.fit(X_train, y_train, validation_split=0.1, verbose=2, batch_size=32, epochs=100)

Epoch 1/100
4/4 - 4s - loss: 1.5864 - acc: 0.2203 - val_loss: 1.6159 - val_acc: 0.0714 - 4s/epoch - 998ms/step
Epoch 2/100
4/4 - 0s - loss: 1.5244 - acc: 0.3814 - val_loss: 1.6406 - val_acc: 0.0714 - 109ms/epoch - 27ms/step
Epoch 3/100
4/4 - 0s - loss: 1.5051 - acc: 0.3136 - val_loss: 1.6758 - val_acc: 0.0714 - 95ms/epoch - 24ms/step
Epoch 4/100
4/4 - 0s - loss: 1.4555 - acc: 0.3644 - val_loss: 1.6571 - val_acc: 0.1429 - 100ms/epoch - 25ms/step
Epoch 5/100
4/4 - 0s - loss: 1.4369 - acc: 0.3644 - val_loss: 1.5896 - val_acc: 0.1429 - 113ms/epoch - 28ms/step
Epoch 6/100
4/4 - 0s - loss: 1.3905 - acc: 0.4576 - val_loss: 1.5366 - val_acc: 0.2143 - 123ms/epoch - 31ms/step
Epoch 7/100
4/4 - 0s - loss: 1.3485 - acc: 0.4068 - val_loss: 1.5823 - val_acc: 0.2143 - 89ms/epoch - 22ms/step
Epoch 8/100
4/4 - 0s - loss: 1.3022 - acc: 0.4153 - val_loss: 1.5330 - val_acc: 0.2857 - 83ms/epoch - 21ms/step
Epoch 9/100
4/4 - 0s - loss: 1.2664 - acc: 0.4492 - val_loss: 1.5187 - val_acc: 0.2143 - 82ms/epoch -

In [ ]:
model_tte2 = Sequential()
model_tte2.add(LSTM(units=128, input_shape=(10, 50), return_sequences=True))
model_tte2.add(Dropout(0.3))
model_tte2.add(LSTM(units=64))
model_tte2.add(Dropout(0.2))
model_tte2.add(Dense(units=32, activation='relu'))
model_tte2.add(Dense(units=20, activation='relu'))
model_tte2.add(Dense(units=5, activation='relu'))

In [ ]:
model_tte2.summary()

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_8 (LSTM)               (None, 10, 128)           91648     
                                                                 
 dropout_8 (Dropout)         (None, 10, 128)           0         
                                                                 
 lstm_9 (LSTM)               (None, 64)                49408     
                                                                 
 dropout_9 (Dropout)         (None, 64)                0         
                                                                 
 dense_10 (Dense)            (None, 32)                2080      
                                                                 
 dense_11 (Dense)            (None, 20)                660       
                                                                 
 dense_12 (Dense)            (None, 5)                

In [ ]:
model_tte2.compile(optimizer='adam', loss=keras.losses.categorical_crossentropy, metrics=['acc'])

In [ ]:
history2 = model_tte2.fit(X_train, y_train, validation_split=0.1, verbose=2, batch_size=32, epochs=100)

Epoch 1/100
4/4 - 4s - loss: 11.0660 - acc: 0.1949 - val_loss: 6.4020 - val_acc: 0.2143 - 4s/epoch - 1s/step
Epoch 2/100
4/4 - 0s - loss: 9.8797 - acc: 0.1525 - val_loss: 6.4158 - val_acc: 0.4286 - 86ms/epoch - 21ms/step
Epoch 3/100
4/4 - 0s - loss: 9.8389 - acc: 0.2119 - val_loss: 6.4190 - val_acc: 0.2857 - 98ms/epoch - 25ms/step
Epoch 4/100
4/4 - 0s - loss: 9.8292 - acc: 0.2203 - val_loss: 6.4013 - val_acc: 0.2857 - 91ms/epoch - 23ms/step
Epoch 5/100
4/4 - 0s - loss: 9.8140 - acc: 0.2373 - val_loss: 6.3589 - val_acc: 0.4286 - 90ms/epoch - 23ms/step
Epoch 6/100
4/4 - 0s - loss: 9.7591 - acc: 0.2627 - val_loss: 6.2710 - val_acc: 0.5000 - 78ms/epoch - 19ms/step
Epoch 7/100
4/4 - 0s - loss: 9.7164 - acc: 0.2881 - val_loss: 6.1765 - val_acc: 0.3571 - 80ms/epoch - 20ms/step
Epoch 8/100
4/4 - 0s - loss: 9.6885 - acc: 0.3136 - val_loss: 6.1641 - val_acc: 0.3571 - 79ms/epoch - 20ms/step
Epoch 9/100
4/4 - 0s - loss: 9.5381 - acc: 0.3305 - val_loss: 6.1576 - val_acc: 0.3571 - 88ms/epoch - 22ms/

In [ ]:
model_tte3 = Sequential()
model_tte3.add(LSTM(units=128, input_shape=(10, 50), return_sequences=True))
model_tte3.add(Dropout(0.3))
model_tte3.add(LSTM(units=64))
model_tte3.add(Dropout(0.2))
model_tte3.add(Dense(units=32, activation='relu'))
model_tte3.add(Dense(units=20, activation='relu'))
model_tte3.add(Dense(units=5, activation='softmax'))

In [ ]:
model_tte3.summary()

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_10 (LSTM)              (None, 10, 128)           91648     
                                                                 
 dropout_10 (Dropout)        (None, 10, 128)           0         
                                                                 
 lstm_11 (LSTM)              (None, 64)                49408     
                                                                 
 dropout_11 (Dropout)        (None, 64)                0         
                                                                 
 dense_13 (Dense)            (None, 32)                2080      
                                                                 
 dense_14 (Dense)            (None, 20)                660       
                                                                 
 dense_15 (Dense)            (None, 5)                

In [ ]:
model_tte3.compile(optimizer='adam', loss=keras.losses.categorical_crossentropy, metrics=['acc'])

In [ ]:
history3 = model_tte3.fit(X_train, y_train, validation_split=0.1, verbose=2, batch_size=32, epochs=100)

Epoch 1/100
4/4 - 4s - loss: 1.6024 - acc: 0.2712 - val_loss: 1.6413 - val_acc: 0.0714 - 4s/epoch - 885ms/step
Epoch 2/100
4/4 - 0s - loss: 1.5616 - acc: 0.3136 - val_loss: 1.7130 - val_acc: 0.0714 - 77ms/epoch - 19ms/step
Epoch 3/100
4/4 - 0s - loss: 1.5349 - acc: 0.3136 - val_loss: 1.7181 - val_acc: 0.0714 - 79ms/epoch - 20ms/step
Epoch 4/100
4/4 - 0s - loss: 1.5013 - acc: 0.3220 - val_loss: 1.6790 - val_acc: 0.0714 - 79ms/epoch - 20ms/step
Epoch 5/100
4/4 - 0s - loss: 1.4594 - acc: 0.3644 - val_loss: 1.6537 - val_acc: 0.1429 - 84ms/epoch - 21ms/step
Epoch 6/100
4/4 - 0s - loss: 1.3895 - acc: 0.4153 - val_loss: 1.6859 - val_acc: 0.1429 - 85ms/epoch - 21ms/step
Epoch 7/100
4/4 - 0s - loss: 1.3023 - acc: 0.4407 - val_loss: 1.5761 - val_acc: 0.2143 - 78ms/epoch - 20ms/step
Epoch 8/100
4/4 - 0s - loss: 1.2198 - acc: 0.5169 - val_loss: 1.5198 - val_acc: 0.2857 - 83ms/epoch - 21ms/step
Epoch 9/100
4/4 - 0s - loss: 1.0703 - acc: 0.5932 - val_loss: 1.2599 - val_acc: 0.6429 - 87ms/epoch - 22m

In [ ]:
test_data = pd.read_csv("/content/test_emoji.csv", header=None)

In [ ]:
test_data.head()

,0,1
0,I want to eat\t,4
1,he did not answer\t,3
2,he got a very nice raise\t,2
3,she got me a nice present\t,2
4,ha ha ha it was so funny\t,2


In [ ]:
# preprocessing test data
test_data[0] = test_data[0].apply(lambda x:x[:-1])
X_test = test_data[0].values
y_test = test_data[1].values

In [ ]:
X_test = get_emb_data(X_test, 10)
y_test = to_categorical(y_test)

In [ ]:
model_tte3.evaluate(X_test, y_test)[1]*100

2/2 [==============================] - 0s 7ms/step - loss: 1.7044 - acc: 0.7321


73.21428656578064

In [ ]:
predicted = model_tte3.predict(X_test)

2/2 [==============================] - 1s 7ms/step


In [ ]:
classes = np.argmax(predicted,axis=1)

In [ ]:
for t in range(len(classes)):
    print(test_data[0].iloc[t])
    print("Predictions: ", emoji.emojize(emoji_mapping[str(classes[t])]))
    print('Actual: ', emoji.emojize(emoji_mapping[str(test_data[1].iloc[t])]))

I want to eat
Predictions:  😋
Actual:  😋
he did not answer
Predictions:  😠
Actual:  😠
he got a very nice raise
Predictions:  💓
Actual:  😁
she got me a nice present
Predictions:  💓
Actual:  😁
ha ha ha it was so funny
Predictions:  😁
Actual:  😁
he is a good friend
Predictions:  😁
Actual:  😁
I am upset
Predictions:  😠
Actual:  😠
We had such a lovely dinner tonight
Predictions:  😁
Actual:  😁
where is the food
Predictions:  😋
Actual:  😋
Stop making this joke ha ha ha
Predictions:  😁
Actual:  😁
where is the ball
Predictions:  ⚾
Actual:  ⚾
work is hard
Predictions:  😁
Actual:  😠
This girl is messing with me
Predictions:  😠
Actual:  😠
are you seriou
Predictions:  😠
Actual:  😠
Let us go play baseball
Predictions:  ⚾
Actual:  ⚾
This stupid grader is not working 
Predictions:  😠
Actual:  😠
work is horrible
Predictions:  😁
Actual:  😠
Congratulation for having a baby
Predictions:  😁
Actual:  😁
stop pissing me of
Predictions:  😠
Actual:  😠
any suggestions for dinner
Predictions:  😁
Actual:  😋
I love

# with VIT

In [ ]:
sample = np.array([caption])
sample = get_emb_data(sample, 10)

In [ ]:
pr = model_tte3.predict(sample)

1/1 [==============================] - 0s 19ms/step


In [ ]:
result = np.argmax(pr,axis=1)

In [ ]:
emoji.emojize(emoji_mapping[str(result[0])])

'💓'